# Gemma 4 smoke test — all DRC language tracks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashuza11/CongoLangBench/blob/main/notebooks/gemma4_all_languages_smoke_test.ipynb)

This notebook runs a small, zero-shot translation smoke test with `google/gemma-4-12B-it` across all 47 frozen language tracks and both translation directions. It saves resumable predictions and BLEU/chrF++ pipeline checks. Smoke-test scores are **not final benchmark results**.

## Before you start

1. In Colab, select **Runtime → Change runtime type → T4 GPU** or a stronger GPU.
2. Accept access to [`google/gemma-4-12B-it`](https://huggingface.co/google/gemma-4-12B-it) with your Hugging Face account.
3. Add an `HF_TOKEN` read token under Colab **Secrets**, or log in when prompted.
4. On your computer, create the private data ZIP from the repository root:

```bash
venv/bin/python scripts/package_colab_benchmarks.py
```

5. Upload `private_data/congolang-benchmark-v1.zip` when this notebook asks for it. The ZIP contains restricted benchmark text: keep it private and never commit or publish it.

In [ ]:
# Evaluation settings
REPO_URL = "https://github.com/Ashuza11/CongoLangBench.git"
REPO_BRANCH = "main"
MODEL_ID = "google/gemma-4-12B-it"
SMOKE_ROWS_PER_DIRECTION = 3  # 47 languages × 2 directions × 3 = 282 requests
MAX_NEW_TOKENS = 256
PROMPT_VERSION = "translation_v1"
assert 1 <= SMOKE_ROWS_PER_DIRECTION <= 10


In [ ]:
# Install current Gemma-compatible libraries in the Colab runtime.
%pip install -q -U transformers accelerate bitsandbytes huggingface_hub sacrebleu pandas

In [ ]:
import platform
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select a GPU runtime and reconnect.")
gpu = torch.cuda.get_device_properties(0)
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB)")


In [ ]:
# Clone the public code and checksum manifests. No restricted text comes from GitHub.
from pathlib import Path
import subprocess

REPO_ROOT = Path("/content/CongoLangBench")
if REPO_ROOT.exists():
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
print(subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip())


## Upload and validate the private benchmark

The next cell uploads the ZIP only into the temporary Colab runtime. It verifies the tracked manifest and every benchmark checksum, and it does not print sentence text.

In [ ]:
from google.colab import files
import csv
import hashlib
import json
import shutil
import zipfile

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("Select private_data/congolang-benchmark-v1.zip from your computer.")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError(f"Upload exactly one ZIP; received: {list(uploaded)}")
bundle_path = Path(zip_names[0])
DATA_ROOT = Path("/content/congolang-benchmark-private")
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True)
with zipfile.ZipFile(bundle_path) as archive:
    for member in archive.infolist():
        destination = (DATA_ROOT / member.filename).resolve()
        if not destination.is_relative_to(DATA_ROOT.resolve()):
            raise ValueError(f"Unsafe ZIP member: {member.filename}")
    archive.extractall(DATA_ROOT)

tracked_manifest = REPO_ROOT / "registry/benchmark_freeze.csv"
private_manifest = DATA_ROOT / "registry/benchmark_freeze.csv"
if file_sha256(tracked_manifest) != file_sha256(private_manifest):
    raise ValueError("Uploaded manifest does not match this Git revision. Rebuild the ZIP.")
with private_manifest.open(encoding="utf-8-sig", newline="") as handle:
    tracks = list(csv.DictReader(handle))
if len(tracks) != 47:
    raise ValueError(f"Expected 47 tracks; found {len(tracks)}")
for track in tracks:
    benchmark = DATA_ROOT / track["benchmark_csv"]
    if not benchmark.is_file():
        raise FileNotFoundError(f"Missing {track['iso_code']} benchmark")
    if file_sha256(benchmark) != track["benchmark_sha256"]:
        raise ValueError(f"Checksum mismatch for {track['iso_code']}")
print(f"Validated {len(tracks)} private tracks / {sum(int(t['benchmark_pairs']) for t in tracks):,} frozen pairs.")
del uploaded  # release the in-memory upload copy


In [ ]:
# Authenticate to Hugging Face. Prefer an HF_TOKEN stored in Colab Secrets.
from huggingface_hub import login, notebook_login
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    notebook_login()


In [ ]:
# Load the instruction-tuned 12B model in 4-bit mode so it fits common Colab GPUs.
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    device_map="auto",
    dtype=compute_dtype,
    low_cpu_mem_usage=True,
)
model.eval()
print(f"Loaded {MODEL_ID} with {compute_dtype} 4-bit inference.")


In [ ]:
# Build a balanced smoke suite: the first N frozen rows in both directions.
import pandas as pd

prompt_template = (REPO_ROOT / "evaluations/prompts/translation_v1.txt").read_text(encoding="utf-8").strip()
jobs = []
for track in tracks:
    frame = pd.read_csv(DATA_ROOT / track["benchmark_csv"], dtype=str, keep_default_na=False).head(SMOKE_ROWS_PER_DIRECTION)
    if len(frame) != SMOKE_ROWS_PER_DIRECTION:
        raise ValueError(f"Not enough smoke rows for {track['iso_code']}")
    for direction in ("reference_to_congolese", "congolese_to_reference"):
        for row in frame.to_dict(orient="records"):
            if direction == "reference_to_congolese":
                source_text, reference_text = row["source_text"], row["target_text"]
                source_language, target_language = track["reference_language"], track["language"]
            else:
                source_text, reference_text = row["target_text"], row["source_text"]
                source_language, target_language = track["language"], track["reference_language"]
            jobs.append({
                "language": track["language"],
                "iso_code": track["iso_code"],
                "direction": direction,
                "record_id": row["record_id"],
                "reference_text": reference_text,
                "prompt": prompt_template.format(
                    source_language=source_language,
                    target_language=target_language,
                    input_text=source_text,
                ),
            })
print(f"Prepared {len(jobs):,} private smoke requests across {len(tracks)} languages and 2 directions.")


In [ ]:
# Run deterministic inference and checkpoint every result. Safe to rerun after interruption.
from datetime import datetime, timezone
import time

RESULT_ROOT = Path("/content/gemma4_all_languages_smoke")
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
PREDICTIONS = RESULT_ROOT / "predictions.jsonl"
completed = {}
if PREDICTIONS.exists():
    for line in PREDICTIONS.read_text(encoding="utf-8").splitlines():
        if line.strip():
            row = json.loads(line)
            completed[(row["iso_code"], row["direction"], row["record_id"])] = row
print(f"Resuming with {len(completed):,}/{len(jobs):,} completed requests.")

last_group = None
for index, job in enumerate(jobs, 1):
    key = (job["iso_code"], job["direction"], job["record_id"])
    if key in completed:
        continue
    group = key[:2]
    if group != last_group:
        print(f"[{index}/{len(jobs)}] {job['language']} ({job['iso_code']}) — {job['direction']}")
        last_group = group
    messages = [{"role": "user", "content": job["prompt"]}]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    ).to(model.device)
    input_length = inputs["input_ids"].shape[-1]
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    prediction = processor.decode(generated[0][input_length:], skip_special_tokens=True).strip()
    result = {
        "model_id": MODEL_ID,
        "iso_code": job["iso_code"],
        "language": job["language"],
        "direction": job["direction"],
        "record_id": job["record_id"],
        "prediction": prediction,
        "elapsed_seconds": round(time.perf_counter() - started, 3),
        "generated_tokens": int(generated.shape[-1] - input_length),
    }
    with PREDICTIONS.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(result, ensure_ascii=False) + "\n")
    completed[key] = result
print(f"Completed {len(completed):,}/{len(jobs):,} requests.")


In [ ]:
# Score each language/direction. These tiny-sample metrics validate the pipeline only.
from collections import defaultdict
from sacrebleu.metrics import BLEU, CHRF

grouped = defaultdict(list)
for job in jobs:
    grouped[(job["language"], job["iso_code"], job["direction"])].append(job)
score_rows = []
bleu_metric = BLEU(effective_order=True)
chrf_metric = CHRF(word_order=2)
for (language, iso_code, direction), group_jobs in grouped.items():
    hypotheses = [completed[(iso_code, direction, job["record_id"])]["prediction"] for job in group_jobs]
    references = [job["reference_text"] for job in group_jobs]
    score_rows.append({
        "language": language,
        "iso_code": iso_code,
        "direction": direction,
        "examples": len(group_jobs),
        "empty_predictions": sum(not text for text in hypotheses),
        "bleu_smoke": bleu_metric.corpus_score(hypotheses, [references]).score,
        "chrf_plus_plus_smoke": chrf_metric.corpus_score(hypotheses, [references]).score,
    })
scores = pd.DataFrame(score_rows).sort_values(["language", "direction"])
scores.to_csv(RESULT_ROOT / "smoke_scores.csv", index=False)
display(scores)
print(f"Scored {len(scores)} language-direction groups; empty predictions: {scores['empty_predictions'].sum()}.")


In [ ]:
# Save reproducibility metadata and download the private result archive.
run_metadata = {
    "run_type": "smoke_test_not_final_benchmark",
    "model_id": MODEL_ID,
    "model_revision": getattr(model.config, "_commit_hash", None),
    "repository_commit": subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip(),
    "benchmark_version": "v1",
    "prompt_version": PROMPT_VERSION,
    "language_tracks": len(tracks),
    "directions": 2,
    "rows_per_direction": SMOKE_ROWS_PER_DIRECTION,
    "requests": len(jobs),
    "do_sample": False,
    "thinking_enabled": False,
    "max_new_tokens": MAX_NEW_TOKENS,
    "quantization": "bitsandbytes_nf4_4bit",
    "gpu": gpu.name,
    "torch_version": torch.__version__,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "contains_model_outputs_from_restricted_inputs": True,
}
(RESULT_ROOT / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n", encoding="utf-8")
archive = shutil.make_archive("/content/gemma4_all_languages_smoke_results", "zip", RESULT_ROOT)
print(f"Created {archive}. Keep the results private until licence handling is confirmed.")
files.download(archive)


## What comes next

Inspect failures and output formatting first. If all 94 language-direction groups complete correctly, increase the sample only after confirming the expected runtime. Final benchmark runs use all 1,500 examples per direction and must be labelled separately from this smoke test. Do not publish uploaded benchmark text or raw outputs from restricted sources.